# Session 03 · MongoDB Operators

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Setup

In [2]:
from campus_data import seed
from mongo_conn import get_db
db = get_db("campus_ops")
print(seed(db))

def names(cursor):
    return [d["name"] for d in cursor]

{'students': 10, 'batches': 3, 'payments': 12}


## Example 1 - comparison and logical

In [3]:
print(names(db.students.find({"marks.python": {"$gte": 80}, "city": {"$in": ["Karachi", "Lahore"]}})))
print(names(db.students.find({"$or": [{"city": "Quetta"}, {"marks.sql": {"$lt": 50}}]})))
print(names(db.students.find({"$nor": [{"city": "Karachi"}, {"city": "Lahore"}]})))
print(names(db.students.find({"marks.python": {"$not": {"$gte": 60}}})))

['Sana Khan', 'Faraz Iqbal']
['Hamza Ali', 'Mariam Javed']
['Ayesha Noor', 'Zoya Hussain', 'Mariam Javed']
['Hamza Ali', 'Usman Tariq']


## Example 2 - element and evaluation

In [4]:
print("no phone:", names(db.students.find({"phone": {"$exists": False}})))
print("name starts with A:", names(db.students.find({"name": {"$regex": "^a", "$options": "i"}})))
print("sql > python:", names(db.students.find({"$expr": {"$gt": ["$marks.sql", "$marks.python"]}})))

no phone: ['Hamza Ali', 'Zoya Hussain', 'Mariam Javed']
name starts with A: ['Ali Raza', 'Ayesha Noor']
sql > python: ['Ayesha Noor', 'Bilal Ahmed', 'Usman Tariq', 'Hina Shah']


## Example 3 - array queries

In [5]:
print("knows python:", len(names(db.students.find({"skills": "python"}))))
print("python AND sql:", names(db.students.find({"skills": {"$all": ["python", "sql"]}})))
print("no skills:", names(db.students.find({"skills": {"$size": 0}})))
db.quizzes.drop()
db.quizzes.insert_many([
    {"name": "Ali", "tests": [{"topic": "sql", "score": 6}, {"topic": "python", "score": 9}]},
    {"name": "Sana", "tests": [{"topic": "sql", "score": 9}, {"topic": "stats", "score": 7}]},
])
print("dot-notation (different elements):", names(db.quizzes.find({"tests.topic": "sql", "tests.score": {"$gte": 8}})))
print("$elemMatch (same element):", names(db.quizzes.find({"tests": {"$elemMatch": {"topic": "sql", "score": {"$gte": 8}}}})))

knows python: 7
python AND sql: ['Sana Khan', 'Zoya Hussain', 'Faraz Iqbal']
no skills: ['Usman Tariq']
dot-notation (different elements): ['Ali', 'Sana']
$elemMatch (same element): ['Sana']


## Example 4 - projections

In [6]:
for d in db.students.find({"batch": "DS-01"}, {"_id": 0, "name": 1, "skills": {"$slice": 1}}):
    print(d)

{'name': 'Ali Raza', 'skills': ['python']}
{'name': 'Sana Khan', 'skills': ['python']}
{'name': 'Usman Tariq', 'skills': []}


## Example 5 - field updates and upsert

In [7]:
db.students.update_one({"student_id": 1}, {"$inc": {"marks.sql": 5}, "$min": {"marks.stats": 60},
                                           "$rename": {"phone": "mobile"}, "$currentDate": {"updated": True}})
print(db.students.find_one({"student_id": 1}, {"_id": 0, "marks": 1, "mobile": 1}))
for _ in range(2):
    db.attendance.update_one({"student_id": 1, "date": "2026-10-27"}, {"$set": {"status": "present"}}, upsert=True)
print("attendance docs:", db.attendance.count_documents({}))

{'marks': {'python': 78, 'sql': 70, 'stats': 60}, 'mobile': '0300-1001111'}
attendance docs: 1


## Example 6 - array updates

In [8]:
db.students.update_one({"student_id": 1}, {"$addToSet": {"skills": "sql"}})
db.students.update_one({"student_id": 1}, {"$addToSet": {"skills": "sql"}})       # no duplicate
db.students.update_many({}, {"$pull": {"skills": "excel"}})
db.quizzes.update_one({"name": "Ali"}, {"$push": {"tests": {"$each": [{"topic": "stats", "score": 8}],
                                                             "$sort": {"score": -1}, "$slice": 2}}})
db.quizzes.update_one({"name": "Ali", "tests.topic": "stats"}, {"$set": {"tests.$.score": 10}})
print(db.students.find_one({"student_id": 1})["skills"], db.students.count_documents({"skills": "excel"}))
print(db.quizzes.find_one({"name": "Ali"}, {"_id": 0}))

['python', 'pandas', 'sql'] 0
{'name': 'Ali', 'tests': [{'topic': 'python', 'score': 9}, {'topic': 'stats', 'score': 10}]}
